In [ ]:
# =========================
# GLOBAL VARS TO GET FROM CONFIG FILE
# =========================

required_vars = [
    "TIMEZONE",
    "OUTPUT_FILES_FOLDER",
    "INPUT_FILES_FOLDER"
]

In [ ]:
# folder for output files
import os
from pathlib import Path

# Create directory if it doesn't exist
OUTPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)




In [ ]:
# import data

from pathlib import Path
import pandas as pd


# Read and concatenate all CSV files
validations = pd.concat(
    (pd.read_csv(csv_file,dtype={12: "string"}) for csv_file in INPUT_FILES_FOLDER.glob("*.csv")),
    ignore_index=True
)

print(validations.shape)


In [ ]:
# create operational_date column

# Convert unix ms → local timezone
created_local = pd.to_datetime(
    validations["created_at"].values,
    unit="ms",
    utc=True
).tz_convert(TIMEZONE)

# Shift operational boundary
created_local = created_local - pd.Timedelta(hours=4)

# Extract YYYYMMDD numerically (no strftime)
validations["operational_date"] = (
    created_local.year * 10_000
    + created_local.month * 100
    + created_local.day
).astype("int32")



In [ ]:
validations.head(2)

In [ ]:
sorted(validations["operational_date"].unique())


In [ ]:
# anonymize
validations["card_serial_number_2"] = (
    validations["card_serial_number"]
    .astype("string")
    .factorize()[0] + 1
)


In [ ]:
# verify anonymization
assert (
    validations.groupby("card_serial_number")["card_serial_number_2"].nunique().max() == 1
)


In [ ]:
#drop identifier column
validations.drop(columns=["card_serial_number"], inplace=True)


In [ ]:
# export


validations.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "validations.csv"),
     index=False
 )